AVISO LEGAL — PROPIEDAD INTELECTUAL

Este material es propiedad intelectual exclusiva de Julián David Flórez Sánchez (LinkedIn). Todos los derechos reservados. Queda terminantemente prohibida su copia, reproducción, distribución, adaptación o uso por cualquier persona natural o jurídica, pública o privada, sin autorización expresa y escrita del autor. El incumplimiento de esta disposición acarrea las sanciones civiles y penales previstas en la Ley 23 de 1982, la Decisión Andina 351 de 1993, los artículos 270-272 del Código Penal Colombiano y los tratados internacionales de propiedad intelectual aplicables. Para solicitar autorización: linkedin.com/in/julianflorezdata

# Guia 05: Redes Neuronales Convolucionales (CNN)

## Electiva II - Deep Learning | Tecnologico de Antioquia

---

**Objetivo:** Comprender la arquitectura y funcionamiento de las Redes Neuronales Convolucionales (CNN) para tareas de vision por computadora, aprendiendo a construir, entrenar y evaluar modelos convolucionales desde cero.

**Conceptos nuevos en esta guia:**
- Operacion de convolucion en imagenes
- Filtros (kernels) y feature maps
- Stride y padding
- Pooling: Max Pooling y Average Pooling
- Arquitectura CNN completa (Conv + Pool + Dense)
- Visualizacion de features aprendidas
- Data Augmentation para imagenes

**Prerrequisito:** Guia 04 - Redes Profundas (DNN) en Problemas Reales

**Duracion estimada:** 3 horas

---

> **Importante - Configuracion de GPU en Colab:** Antes de comenzar, activa la GPU en Google Colab para acelerar el entrenamiento de las CNN:
> 1. Ve a **Entorno de ejecucion** (Runtime)
> 2. Selecciona **Cambiar tipo de entorno de ejecucion** (Change runtime type)
> 3. En **Acelerador de hardware** selecciona **GPU T4**
> 4. Haz clic en **Guardar**

> **Aviso de evaluacion** \u270d\ufe0f: A lo largo de esta guia encontraras preguntas marcadas con \u270d\ufe0f que debes responder. Estas preguntas seran evaluadas y forman parte de tu nota. Responde de manera completa y reflexiva.

---
## 1. Configuracion del Entorno

Importamos todas las librerias necesarias, verificamos la GPU y fijamos la semilla para reproducibilidad.

In [ ]:
# ============================================================
# Importaciones principales
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

# TensorFlow y Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Para convolucion manual
from scipy import signal

# Utilidades
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# Semilla para reproducibilidad
# ============================================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Configuracion de graficos
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print("Entorno configurado correctamente.")

In [ ]:
# ============================================================
# Verificar disponibilidad de GPU
# ============================================================
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(f"GPU(s) disponible(s): {len(gpus)}")
    for gpu in gpus:
        print(f"  - {gpu.name}")
    # Permitir crecimiento de memoria para evitar errores
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("\nGPU configurada correctamente. El entrenamiento sera mas rapido.")
else:
    print("No se detecto GPU. El entrenamiento sera mas lento.")
    print("Si estas en Colab: Entorno de ejecucion -> Cambiar tipo -> GPU T4")

---
## 2. Marco Teorico

### 2.1 ¿Por que las DNN no son ideales para imagenes?

En las guias anteriores usamos redes densas (DNN) para clasificar imagenes de MNIST y Fashion-MNIST. Si bien funcionaron razonablemente bien, las **redes completamente conectadas** tienen serios problemas cuando se aplican a imagenes:

**Problema 1: Demasiados parametros**

Consideremos una imagen a color de apenas 32x32 pixeles (muy pequena). Tiene $32 \times 32 \times 3 = 3{,}072$ valores de entrada. Si la primera capa oculta tiene 512 neuronas, eso implica:

$$\text{Parametros} = 3{,}072 \times 512 + 512 = 1{,}573{,}376 \text{ parametros solo en la primera capa}$$

Para imagenes mas realistas de 224x224x3 (como las de ImageNet):

$$\text{Parametros} = 224 \times 224 \times 3 \times 512 + 512 = 77{,}071{,}360 \approx 77 \text{ millones de parametros}$$

Esto es computacionalmente prohibitivo y propenso al sobreajuste.

**Problema 2: No captura patrones espaciales**

Cuando aplanamos una imagen con `Flatten()`, destruimos toda la informacion espacial. Un gato en la esquina superior izquierda y el mismo gato en el centro serian patrones completamente diferentes para una DNN, aunque es el mismo gato. Las DNN no entienden que los pixeles vecinos estan relacionados.

**Problema 3: No es invariante a la posicion**

Si entrenamos una DNN para reconocer un gato que siempre esta centrado, no podra reconocer el mismo gato si aparece desplazado en la imagen.

**Solucion: Las Redes Neuronales Convolucionales (CNN)**

Las CNN resuelven estos tres problemas usando:
- **Conectividad local:** cada neurona solo mira una region pequena de la imagen
- **Pesos compartidos:** el mismo filtro se aplica en toda la imagen
- **Jerarquia de features:** las primeras capas detectan bordes, las siguientes formas, y las ultimas objetos complejos

### 2.2 ¿Que es una convolucion?

La **convolucion** es una operacion matematica que combina dos funciones para producir una tercera. En el contexto de imagenes, consiste en deslizar un filtro (kernel) sobre la imagen y calcular la suma ponderada en cada posicion.

#### Ejemplo paso a paso

Supongamos que tenemos una imagen de 5x5 pixeles (en escala de grises) y un filtro de 3x3:

**Imagen (5x5):**
```
| 1 | 2 | 0 | 1 | 3 |
| 0 | 1 | 2 | 3 | 1 |
| 1 | 0 | 1 | 0 | 2 |
| 2 | 1 | 0 | 1 | 0 |
| 0 | 3 | 2 | 1 | 1 |
```

**Filtro/Kernel (3x3) - Detector de bordes horizontales:**
```
| -1 | -1 | -1 |
|  0 |  0 |  0 |
|  1 |  1 |  1 |
```

**Calculo en la posicion (0,0):**

El filtro se coloca sobre la esquina superior izquierda de la imagen (pixeles en las filas 0-2 y columnas 0-2):

$$\text{Resultado}_{(0,0)} = (1 \times -1) + (2 \times -1) + (0 \times -1) + (0 \times 0) + (1 \times 0) + (2 \times 0) + (1 \times 1) + (0 \times 1) + (1 \times 1) = -3 + 0 + 2 = -1$$

Luego se desliza una posicion a la derecha y se repite el calculo. Se continua hasta cubrir toda la imagen.

El **resultado** es un **feature map** (mapa de caracteristicas) de 3x3 (la imagen se reduce porque el filtro no puede ir mas alla de los bordes).

#### Formula general de la convolucion 2D

Para una imagen $I$ y un filtro $K$ de tamano $f \times f$:

$$(I * K)(i, j) = \sum_{m=0}^{f-1} \sum_{n=0}^{f-1} I(i+m, j+n) \cdot K(m, n)$$

donde $(i, j)$ es la posicion en el feature map de salida.

### 2.3 Filtros/Kernels: detectores de caracteristicas

Los filtros son la clave de las CNN. Cada filtro es una pequena matriz de pesos que detecta un patron especifico:

| Filtro | Proposito | Ejemplo 3x3 |
|--------|-----------|-------------|
| Bordes horizontales | Detecta transiciones arriba/abajo | `[[-1,-1,-1],[0,0,0],[1,1,1]]` |
| Bordes verticales | Detecta transiciones izquierda/derecha | `[[-1,0,1],[-1,0,1],[-1,0,1]]` |
| Bordes diagonales | Detecta transiciones en diagonal | `[[-1,-1,0],[-1,0,1],[0,1,1]]` |
| Desenfoque (blur) | Suaviza la imagen | `[[1,1,1],[1,1,1],[1,1,1]] / 9` |
| Nitidez (sharpen) | Realza detalles | `[[0,-1,0],[-1,5,-1],[0,-1,0]]` |

**Punto clave:** En las CNN, los valores de los filtros **NO se definen manualmente**. La red los **aprende automaticamente** durante el entrenamiento mediante backpropagation. Cada capa convolucional tiene multiples filtros, y cada uno aprende a detectar un patron diferente.

### 2.4 Feature Maps

El resultado de aplicar un filtro a una imagen se llama **feature map** (o mapa de activacion). Si una capa convolucional tiene $n$ filtros, producira $n$ feature maps.

Por ejemplo, si aplicamos 32 filtros de 3x3 a una imagen de 32x32, obtenemos 32 feature maps de 30x30 (sin padding). Cada feature map resalta un aspecto diferente de la imagen.

### 2.5 Stride y Padding

#### Stride (paso)

El **stride** define cuantos pixeles se desplaza el filtro en cada paso:
- **Stride = 1:** el filtro se mueve de a 1 pixel (por defecto)
- **Stride = 2:** el filtro se mueve de a 2 pixeles (reduce la salida a la mitad)

#### Padding (relleno)

El **padding** agrega pixeles (generalmente ceros) alrededor de la imagen:
- **Valid (sin padding):** el filtro solo se aplica donde cabe completamente. La salida es mas pequena.
- **Same (con padding):** se agregan ceros alrededor de la imagen para que la salida tenga el **mismo tamano** que la entrada.

#### Formula del tamano de salida

Para una entrada de tamano $n$, filtro de tamano $f$, stride $s$ y padding $p$:

$$\text{Tamano de salida} = \left\lfloor \frac{n + 2p - f}{s} \right\rfloor + 1$$

**Ejemplos:**

| Entrada ($n$) | Filtro ($f$) | Stride ($s$) | Padding ($p$) | Salida |
|:---:|:---:|:---:|:---:|:---:|
| 32 | 3 | 1 | 0 (valid) | $\lfloor(32+0-3)/1\rfloor+1 = 30$ |
| 32 | 3 | 1 | 1 (same) | $\lfloor(32+2-3)/1\rfloor+1 = 32$ |
| 32 | 3 | 2 | 0 (valid) | $\lfloor(32+0-3)/2\rfloor+1 = 15$ |
| 32 | 5 | 1 | 0 (valid) | $\lfloor(32+0-5)/1\rfloor+1 = 28$ |
| 32 | 5 | 1 | 2 (same) | $\lfloor(32+4-5)/1\rfloor+1 = 32$ |

### 2.6 Pooling: reduccion de dimensionalidad

Las capas de **pooling** reducen el tamano espacial de los feature maps, lo que:
- Reduce la cantidad de parametros y computo
- Proporciona cierta invariancia a pequenas traslaciones
- Ayuda a prevenir el sobreajuste

#### Max Pooling

Toma el **valor maximo** en cada ventana. Es el tipo de pooling mas usado.

```
Entrada (4x4):              MaxPool 2x2, stride 2:
| 1 | 3 | 2 | 1 |          | 5 | 3 |
| 5 | 2 | 0 | 2 |   --->   | 4 | 7 |
| 1 | 4 | 3 | 7 |
| 2 | 0 | 1 | 4 |
```

Para la ventana superior izquierda: $\max(1, 3, 5, 2) = 5$

#### Average Pooling

Toma el **valor promedio** en cada ventana.

```
Entrada (4x4):              AvgPool 2x2, stride 2:
| 1 | 3 | 2 | 1 |          | 2.75 | 1.25 |
| 5 | 2 | 0 | 2 |   --->   | 1.75 | 3.75 |
| 1 | 4 | 3 | 7 |
| 2 | 0 | 1 | 4 |
```

Para la ventana superior izquierda: $(1 + 3 + 5 + 2) / 4 = 2.75$

**¿Por que Max Pooling es mas popular?** Porque preserva las activaciones mas fuertes (los patrones mas relevantes detectados por los filtros), mientras que Average Pooling diluye la senal al promediar.

### 2.7 Arquitectura CNN tipica

Una CNN clasica sigue el patron:

```
IMAGEN
  |
  v
[Conv2D] --> [Activacion ReLU] --> [MaxPooling2D]     <-- Bloque convolucional 1
  |
  v
[Conv2D] --> [Activacion ReLU] --> [MaxPooling2D]     <-- Bloque convolucional 2
  |
  v
[Conv2D] --> [Activacion ReLU] --> [MaxPooling2D]     <-- Bloque convolucional 3
  |
  v
[Flatten]                                              <-- Aplanar a 1D
  |
  v
[Dense] --> [Activacion ReLU] --> [Dropout]            <-- Capa densa
  |
  v
[Dense con Softmax]                                    <-- Capa de salida
```

**Parte convolucional (extractor de caracteristicas):** Las capas Conv2D + MaxPooling extraen caracteristicas de la imagen, desde bordes simples hasta patrones complejos.

**Parte densa (clasificador):** Las capas Dense clasifican las caracteristicas extraidas en las clases finales.

### 2.8 Parametros: capa convolucional vs capa densa

Una de las grandes ventajas de las CNN es que tienen **muchos menos parametros** que las DNN:

**Capa densa (Dense)** con entrada de 32x32x3 = 3072 y 128 neuronas:
$$\text{Parametros} = 3{,}072 \times 128 + 128 = 393{,}344$$

**Capa convolucional (Conv2D)** con 32 filtros de 3x3 y entrada de 3 canales:
$$\text{Parametros} = (3 \times 3 \times 3) \times 32 + 32 = 896$$

La capa convolucional tiene **439 veces menos parametros** que la densa, pero puede extraer caracteristicas mucho mas relevantes de la imagen. Esta eficiencia se debe al **peso compartido**: el mismo filtro se aplica en todas las posiciones de la imagen.

---
## 3. Carga y Exploracion de Datos

Trabajaremos con el dataset **CIFAR-10**, que contiene 60,000 imagenes a color de 32x32 pixeles en 10 clases. Es significativamente mas complejo que MNIST y Fashion-MNIST.

**Las 10 clases de CIFAR-10:**
avion (airplane), automovil (automobile), pajaro (bird), gato (cat), venado (deer), perro (dog), rana (frog), caballo (horse), barco (ship), camion (truck)

In [ ]:
# ============================================================
# Cargar el dataset CIFAR-10
# ============================================================
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.cifar10.load_data()

# Nombres de las clases en espanol
nombres_clases = ['Avion', 'Automovil', 'Pajaro', 'Gato', 'Venado',
                  'Perro', 'Rana', 'Caballo', 'Barco', 'Camion']

print("Dataset CIFAR-10 cargado.")
print(f"  Entrenamiento: {X_train_full.shape[0]} imagenes")
print(f"  Test:          {X_test.shape[0]} imagenes")
print(f"  Tamano de imagen: {X_train_full.shape[1:]} (alto x ancho x canales)")
print(f"  Tipo de datos: {X_train_full.dtype}")
print(f"  Rango de pixeles: [{X_train_full.min()}, {X_train_full.max()}]")
print(f"  Numero de clases: {len(nombres_clases)}")

# Comparacion con datasets anteriores
print("\n" + "=" * 60)
print("COMPARACION CON DATASETS ANTERIORES")
print("=" * 60)
print(f"  {'Dataset':<20s} {'Tamano':<15s} {'Canales':<10s} {'Clases':<8s} {'Pixeles totales'}")
print(f"  {'MNIST':<20s} {'28x28':<15s} {'1 (gris)':<10s} {'10':<8s} {28*28*1}")
print(f"  {'Fashion-MNIST':<20s} {'28x28':<15s} {'1 (gris)':<10s} {'10':<8s} {28*28*1}")
print(f"  {'CIFAR-10':<20s} {'32x32':<15s} {'3 (RGB)':<10s} {'10':<8s} {32*32*3}")
print("\n  CIFAR-10 tiene imagenes a COLOR y es mucho mas dificil de clasificar.")

In [ ]:
# ============================================================
# Visualizar imagenes de ejemplo por clase
# ============================================================
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()

for clase_idx in range(10):
    # Encontrar indices de imagenes de esta clase
    indices = np.where(y_train_full.flatten() == clase_idx)[0]
    # Seleccionar una imagen aleatoria
    img_idx = indices[np.random.randint(len(indices))]

    axes[clase_idx].imshow(X_train_full[img_idx])
    axes[clase_idx].set_title(f'Clase {clase_idx}: {nombres_clases[clase_idx]}', fontsize=11)
    axes[clase_idx].axis('off')

plt.suptitle('Una imagen de ejemplo por clase - CIFAR-10', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Grid de imagenes: 5 ejemplos por clase
# ============================================================
fig, axes = plt.subplots(10, 5, figsize=(12, 22))

for clase_idx in range(10):
    indices = np.where(y_train_full.flatten() == clase_idx)[0]
    seleccion = np.random.choice(indices, 5, replace=False)

    for j, img_idx in enumerate(seleccion):
        axes[clase_idx, j].imshow(X_train_full[img_idx])
        axes[clase_idx, j].axis('off')
        if j == 0:
            axes[clase_idx, j].set_ylabel(nombres_clases[clase_idx], fontsize=11,
                                           rotation=0, labelpad=60, va='center')

plt.suptitle('5 ejemplos por clase - CIFAR-10', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Observa la variabilidad DENTRO de cada clase:")
print("- Los objetos aparecen en diferentes posiciones, tamanos y orientaciones")
print("- Los fondos son variados y complejos")
print("- La resolucion es baja (32x32), lo que dificulta la clasificacion")

In [ ]:
# ============================================================
# Distribucion de clases
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Entrenamiento
conteo_train = np.bincount(y_train_full.flatten())
axes[0].bar(range(10), conteo_train, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_xticks(range(10))
axes[0].set_xticklabels(nombres_clases, rotation=45, ha='right')
axes[0].set_ylabel('Cantidad')
axes[0].set_title('Distribucion de clases - Entrenamiento')
for i, v in enumerate(conteo_train):
    axes[0].text(i, v + 50, str(v), ha='center', fontsize=9)

# Test
conteo_test = np.bincount(y_test.flatten())
axes[1].bar(range(10), conteo_test, color='coral', edgecolor='black', alpha=0.8)
axes[1].set_xticks(range(10))
axes[1].set_xticklabels(nombres_clases, rotation=45, ha='right')
axes[1].set_ylabel('Cantidad')
axes[1].set_title('Distribucion de clases - Test')
for i, v in enumerate(conteo_test):
    axes[1].text(i, v + 20, str(v), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("El dataset esta perfectamente BALANCEADO: 5,000 imagenes por clase en train, 1,000 en test.")

In [ ]:
# ============================================================
# Normalizar los datos a [0, 1]
# ============================================================
X_train_full = X_train_full.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

print(f"Rango de pixeles despues de normalizar: [{X_train_full.min()}, {X_train_full.max()}]")
print(f"Tipo de datos: {X_train_full.dtype}")

# Separar conjunto de validacion del entrenamiento
# Usamos las ultimas 5000 imagenes como validacion
X_val = X_train_full[-5000:]
y_val = y_train_full[-5000:]
X_train = X_train_full[:-5000]
y_train = y_train_full[:-5000]

print(f"\nDivision de datos:")
print(f"  Entrenamiento: {X_train.shape[0]} imagenes")
print(f"  Validacion:    {X_val.shape[0]} imagenes")
print(f"  Test:          {X_test.shape[0]} imagenes")

---
## 4. Experimentacion Guiada

### Experimento 1: Visualizar convoluciones manualmente

Antes de dejar que la CNN aprenda sus propios filtros, vamos a aplicar filtros **manualmente** usando NumPy para entender visualmente que hace la convolucion.

In [ ]:
# ============================================================
# EXPERIMENTO 1: Convoluciones manuales con filtros predefinidos
# ============================================================

# Seleccionar una imagen del dataset
img_ejemplo = X_train[7]  # Seleccionamos una imagen

# Convertir a escala de grises para simplificar (promedio de los 3 canales)
img_gris = np.mean(img_ejemplo, axis=2)

print(f"Imagen original (color): {img_ejemplo.shape}")
print(f"Imagen en escala de grises: {img_gris.shape}")

# Definir filtros manuales
filtros = {
    'Bordes horizontales': np.array([
        [-1, -1, -1],
        [ 0,  0,  0],
        [ 1,  1,  1]
    ], dtype='float32'),

    'Bordes verticales': np.array([
        [-1, 0, 1],
        [-1, 0, 1],
        [-1, 0, 1]
    ], dtype='float32'),

    'Bordes diagonales': np.array([
        [-1, -1,  0],
        [-1,  0,  1],
        [ 0,  1,  1]
    ], dtype='float32'),

    'Desenfoque (blur)': np.array([
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1]
    ], dtype='float32') / 9.0,

    'Nitidez (sharpen)': np.array([
        [ 0, -1,  0],
        [-1,  5, -1],
        [ 0, -1,  0]
    ], dtype='float32'),

    'Detector de bordes (Laplaciano)': np.array([
        [ 0, -1,  0],
        [-1,  4, -1],
        [ 0, -1,  0]
    ], dtype='float32')
}

print(f"\nFiltros definidos: {list(filtros.keys())}")

In [ ]:
# Aplicar cada filtro y visualizar
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Imagen original en color
axes[0, 0].imshow(img_ejemplo)
axes[0, 0].set_title('Original (color)', fontsize=12)
axes[0, 0].axis('off')

# Imagen en escala de grises
axes[0, 1].imshow(img_gris, cmap='gray')
axes[0, 1].set_title('Escala de grises', fontsize=12)
axes[0, 1].axis('off')

# Aplicar cada filtro usando scipy.signal.convolve2d
posiciones = [(0, 2), (0, 3), (1, 0), (1, 1), (1, 2), (1, 3)]

for idx, (nombre, filtro) in enumerate(filtros.items()):
    fila, col = posiciones[idx]

    # Aplicar convolucion 2D
    resultado = signal.convolve2d(img_gris, filtro, mode='valid')

    axes[fila, col].imshow(resultado, cmap='gray')
    axes[fila, col].set_title(nombre, fontsize=11)
    axes[fila, col].axis('off')

plt.suptitle('Efecto de diferentes filtros sobre la imagen', fontsize=14)
plt.tight_layout()
plt.show()

print("Observaciones:")
print("- El filtro de bordes horizontales resalta las transiciones de arriba a abajo")
print("- El filtro de bordes verticales resalta las transiciones de izquierda a derecha")
print("- El filtro de desenfoque suaviza la imagen (promedia pixeles vecinos)")
print("- El filtro de nitidez realza los detalles y bordes")
print("- El Laplaciano detecta bordes en todas las direcciones")

### \u270d\ufe0f Pregunta - Experimento 1

**¿Que tipo de informacion captura cada filtro? ¿Por que el filtro de bordes horizontales no detecta bordes verticales?**

*Pista: Observa los valores del filtro de bordes horizontales: la fila superior es negativa, la del medio es cero y la inferior es positiva. ¿Que pasa cuando se aplica a una zona donde todos los pixeles son iguales? ¿Y donde hay un cambio brusco de arriba a abajo?*

### \u270d\ufe0f Tu respuesta:

*Escribe aqui tu respuesta...*

---
### Experimento 2: DNN vs CNN en CIFAR-10

Vamos a comparar directamente una red densa (DNN) contra una red convolucional (CNN) basica en el mismo dataset para ver la diferencia en rendimiento y eficiencia.

In [ ]:
# ============================================================
# EXPERIMENTO 2: DNN vs CNN en CIFAR-10
# ============================================================
import time

# ---- Modelo 1: DNN pura (baseline) ----
print("=" * 60)
print("MODELO 1: DNN pura (Flatten + Dense)")
print("=" * 60)

modelo_dnn = keras.Sequential([
    layers.Flatten(input_shape=(32, 32, 3)),   # Aplanar la imagen: 32*32*3 = 3072
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='DNN_baseline')

modelo_dnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Parametros totales DNN: {modelo_dnn.count_params():,}")

# Entrenar DNN
inicio_dnn = time.time()
historia_dnn = modelo_dnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=64,
    verbose=0,
    callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)
tiempo_dnn = time.time() - inicio_dnn

# Evaluar DNN
_, acc_dnn = modelo_dnn.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy en test: {acc_dnn:.4f} ({acc_dnn*100:.1f}%)")
print(f"Tiempo de entrenamiento: {tiempo_dnn:.1f} segundos")
print(f"Epocas entrenadas: {len(historia_dnn.history['loss'])}")

In [ ]:
# ---- Modelo 2: CNN basica ----
print("=" * 60)
print("MODELO 2: CNN basica (2 bloques Conv2D + MaxPool)")
print("=" * 60)

modelo_cnn = keras.Sequential([
    # Bloque convolucional 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),

    # Bloque convolucional 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Clasificador
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='CNN_basica')

modelo_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Parametros totales CNN: {modelo_cnn.count_params():,}")

# Entrenar CNN
inicio_cnn = time.time()
historia_cnn = modelo_cnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=64,
    verbose=0,
    callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)
tiempo_cnn = time.time() - inicio_cnn

# Evaluar CNN
_, acc_cnn = modelo_cnn.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy en test: {acc_cnn:.4f} ({acc_cnn*100:.1f}%)")
print(f"Tiempo de entrenamiento: {tiempo_cnn:.1f} segundos")
print(f"Epocas entrenadas: {len(historia_cnn.history['loss'])}")

In [ ]:
# ---- Comparacion DNN vs CNN ----
print("\n" + "=" * 60)
print("COMPARACION: DNN vs CNN")
print("=" * 60)
print(f"{'Metrica':<25s} {'DNN':>12s} {'CNN':>12s} {'Diferencia':>12s}")
print("-" * 61)
print(f"{'Parametros':<25s} {modelo_dnn.count_params():>12,} {modelo_cnn.count_params():>12,} {modelo_dnn.count_params() - modelo_cnn.count_params():>+12,}")
print(f"{'Accuracy test':<25s} {acc_dnn:>12.4f} {acc_cnn:>12.4f} {acc_cnn - acc_dnn:>+12.4f}")
print(f"{'Tiempo (seg)':<25s} {tiempo_dnn:>12.1f} {tiempo_cnn:>12.1f} {tiempo_cnn - tiempo_dnn:>+12.1f}")
print(f"{'Epocas':<25s} {len(historia_dnn.history['loss']):>12d} {len(historia_cnn.history['loss']):>12d}")

# Graficar comparacion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(historia_dnn.history['val_accuracy'], label='DNN', linewidth=2, color='#e74c3c')
axes[0].plot(historia_cnn.history['val_accuracy'], label='CNN', linewidth=2, color='#2ecc71')
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy en Validacion: DNN vs CNN')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(historia_dnn.history['val_loss'], label='DNN', linewidth=2, color='#e74c3c')
axes[1].plot(historia_cnn.history['val_loss'], label='CNN', linewidth=2, color='#2ecc71')
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss en Validacion: DNN vs CNN')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### \u270d\ufe0f Pregunta - Experimento 2

**¿Cuantos parametros tiene la DNN vs la CNN? ¿Cual tiene mejor accuracy? ¿Por que la CNN es mas eficiente (mejor rendimiento con menos parametros)?**

*Pista: Piensa en los conceptos de conectividad local y pesos compartidos que vimos en el marco teorico.*

### \u270d\ufe0f Tu respuesta:

*Escribe aqui tu respuesta...*

---
### Experimento 3: Profundidad de la CNN

¿Que pasa si agregamos mas capas convolucionales? Vamos a comparar CNN con 1, 2 y 3 bloques convolucionales, incrementando el numero de filtros en cada bloque.

In [ ]:
# ============================================================
# EXPERIMENTO 3: Profundidad de la CNN
# ============================================================

def crear_cnn_bloques(n_bloques, nombre):
    """
    Crea una CNN con n bloques convolucionales.
    Cada bloque: Conv2D + ReLU + MaxPooling2D
    Los filtros se duplican en cada bloque: 32 -> 64 -> 128
    """
    modelo = keras.Sequential(name=nombre)

    filtros_por_bloque = [32, 64, 128]

    for i in range(n_bloques):
        n_filtros = filtros_por_bloque[i] if i < len(filtros_por_bloque) else 128

        if i == 0:
            modelo.add(layers.Conv2D(n_filtros, (3, 3), activation='relu',
                                     padding='same', input_shape=(32, 32, 3)))
        else:
            modelo.add(layers.Conv2D(n_filtros, (3, 3), activation='relu', padding='same'))

        modelo.add(layers.MaxPooling2D((2, 2)))

    modelo.add(layers.Flatten())
    modelo.add(layers.Dense(128, activation='relu'))
    modelo.add(layers.Dense(10, activation='softmax'))

    return modelo

# Crear las tres variantes
modelos_profundidad = {
    '1 bloque (32)': crear_cnn_bloques(1, 'CNN_1bloque'),
    '2 bloques (32-64)': crear_cnn_bloques(2, 'CNN_2bloques'),
    '3 bloques (32-64-128)': crear_cnn_bloques(3, 'CNN_3bloques')
}

# Mostrar resumen de cada modelo
for nombre, modelo in modelos_profundidad.items():
    modelo.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print(f"{nombre}: {modelo.count_params():,} parametros")

In [ ]:
# Entrenar las tres variantes
historias_prof = {}
resultados_prof = {}

for nombre, modelo in modelos_profundidad.items():
    print(f"\nEntrenando: {nombre}")
    print("-" * 40)

    inicio = time.time()
    historia = modelo.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=64,
        verbose=0,
        callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
    )
    tiempo = time.time() - inicio

    historias_prof[nombre] = historia

    _, acc = modelo.evaluate(X_test, y_test, verbose=0)
    resultados_prof[nombre] = {
        'Accuracy': acc,
        'Parametros': modelo.count_params(),
        'Epocas': len(historia.history['loss']),
        'Tiempo': tiempo
    }

    print(f"  Accuracy: {acc:.4f} | Parametros: {modelo.count_params():,} | Tiempo: {tiempo:.1f}s")

print("\nEntrenamiento completado.")

In [ ]:
# Graficar comparacion de profundidad
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colores = ['#e74c3c', '#2ecc71', '#3498db']

for i, (nombre, historia) in enumerate(historias_prof.items()):
    axes[0].plot(historia.history['val_accuracy'], label=nombre, linewidth=2, color=colores[i])
    axes[1].plot(historia.history['val_loss'], label=nombre, linewidth=2, color=colores[i])

axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy en Validacion vs Profundidad')
axes[0].legend()
axes[0].grid(True)

axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss en Validacion vs Profundidad')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Tabla de resultados
print("\n" + "=" * 70)
print("COMPARACION POR PROFUNDIDAD")
print("=" * 70)
print(f"{'Modelo':<25s} {'Accuracy':>10s} {'Parametros':>12s} {'Tiempo':>10s}")
print("-" * 57)
for nombre, res in resultados_prof.items():
    print(f"{nombre:<25s} {res['Accuracy']:>10.4f} {res['Parametros']:>12,} {res['Tiempo']:>10.1f}s")

### \u270d\ufe0f Pregunta - Experimento 3

**¿Que efecto tuvo agregar mas bloques convolucionales? ¿Los filtros iniciales y finales detectan el mismo tipo de patrones? Explica por que las CNN mas profundas suelen funcionar mejor.**

*Pista: Las primeras capas detectan patrones simples (bordes), las intermedias patrones mas complejos (texturas, formas) y las ultimas patrones de alto nivel (partes de objetos).*

### \u270d\ufe0f Tu respuesta:

*Escribe aqui tu respuesta...*

---
### Experimento 4: Efecto del tamano del filtro

¿Importa el tamano del filtro? Vamos a comparar filtros de 3x3, 5x5 y 7x7 en una misma arquitectura CNN.

In [ ]:
# ============================================================
# EXPERIMENTO 4: Efecto del tamano del filtro
# ============================================================

def crear_cnn_filtro(tamano_filtro, nombre):
    """Crea una CNN con 2 bloques usando el tamano de filtro especificado."""
    modelo = keras.Sequential([
        layers.Conv2D(32, (tamano_filtro, tamano_filtro), activation='relu',
                     padding='same', input_shape=(32, 32, 3)),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (tamano_filtro, tamano_filtro), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ], name=nombre)
    return modelo

# Crear modelos con diferentes tamanos de filtro
modelos_filtro = {
    'Filtro 3x3': crear_cnn_filtro(3, 'CNN_3x3'),
    'Filtro 5x5': crear_cnn_filtro(5, 'CNN_5x5'),
    'Filtro 7x7': crear_cnn_filtro(7, 'CNN_7x7')
}

# Compilar y mostrar parametros
for nombre, modelo in modelos_filtro.items():
    modelo.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print(f"{nombre}: {modelo.count_params():,} parametros")

In [ ]:
# Entrenar los tres modelos
historias_filtro = {}
resultados_filtro = {}

for nombre, modelo in modelos_filtro.items():
    print(f"\nEntrenando: {nombre}")
    print("-" * 40)

    inicio = time.time()
    historia = modelo.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=64,
        verbose=0,
        callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
    )
    tiempo = time.time() - inicio

    historias_filtro[nombre] = historia
    _, acc = modelo.evaluate(X_test, y_test, verbose=0)
    resultados_filtro[nombre] = {'Accuracy': acc, 'Parametros': modelo.count_params(), 'Tiempo': tiempo}

    print(f"  Accuracy: {acc:.4f} | Parametros: {modelo.count_params():,} | Tiempo: {tiempo:.1f}s")

# Graficar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colores = ['#e74c3c', '#2ecc71', '#3498db']

for i, (nombre, historia) in enumerate(historias_filtro.items()):
    axes[0].plot(historia.history['val_accuracy'], label=nombre, linewidth=2, color=colores[i])
    axes[1].plot(historia.history['val_loss'], label=nombre, linewidth=2, color=colores[i])

axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs Tamano de Filtro')
axes[0].legend()
axes[0].grid(True)

axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss vs Tamano de Filtro')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Tabla
print("\n" + "=" * 65)
print("COMPARACION POR TAMANO DE FILTRO")
print("=" * 65)
print(f"{'Filtro':<15s} {'Accuracy':>10s} {'Parametros':>12s} {'Tiempo':>10s}")
print("-" * 47)
for nombre, res in resultados_filtro.items():
    print(f"{nombre:<15s} {res['Accuracy']:>10.4f} {res['Parametros']:>12,} {res['Tiempo']:>10.1f}s")

### \u270d\ufe0f Pregunta - Experimento 4

**¿Que tamano de filtro dio el mejor resultado? ¿Cual es el trade-off entre filtros grandes y pequenos? ¿Por que los filtros 3x3 son los mas populares en las CNN modernas?**

*Pista: Dos capas con filtros 3x3 en cascada cubren un campo receptivo de 5x5, pero con menos parametros que un unico filtro de 5x5.*

### \u270d\ufe0f Tu respuesta:

*Escribe aqui tu respuesta...*

---
### Experimento 5: Max Pooling vs Average Pooling vs Stride

Hay varias formas de reducir la dimensionalidad espacial en una CNN. Vamos a comparar tres enfoques:
1. **Max Pooling:** toma el maximo de cada ventana
2. **Average Pooling:** toma el promedio de cada ventana
3. **Stride = 2 en Conv2D:** la propia convolucion reduce el tamano (sin capa de pooling)

In [ ]:
# ============================================================
# EXPERIMENTO 5: Metodos de reduccion de dimensionalidad
# ============================================================

# Modelo con Max Pooling
modelo_maxpool = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='CNN_MaxPool')

# Modelo con Average Pooling
modelo_avgpool = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.AveragePooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.AveragePooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='CNN_AvgPool')

# Modelo con Stride=2 (sin pooling)
modelo_stride = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', strides=2, input_shape=(32, 32, 3)),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same', strides=2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='CNN_Stride2')

modelos_pool = {
    'Max Pooling': modelo_maxpool,
    'Average Pooling': modelo_avgpool,
    'Stride = 2': modelo_stride
}

for nombre, modelo in modelos_pool.items():
    modelo.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print(f"{nombre}: {modelo.count_params():,} parametros")

In [ ]:
# Entrenar los tres modelos
historias_pool = {}
resultados_pool = {}

for nombre, modelo in modelos_pool.items():
    print(f"\nEntrenando: {nombre}")
    print("-" * 40)

    inicio = time.time()
    historia = modelo.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=64,
        verbose=0,
        callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
    )
    tiempo = time.time() - inicio

    historias_pool[nombre] = historia
    _, acc = modelo.evaluate(X_test, y_test, verbose=0)
    resultados_pool[nombre] = {'Accuracy': acc, 'Parametros': modelo.count_params(), 'Tiempo': tiempo}

    print(f"  Accuracy: {acc:.4f} | Parametros: {modelo.count_params():,} | Tiempo: {tiempo:.1f}s")

# Graficar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colores = ['#e74c3c', '#2ecc71', '#3498db']

for i, (nombre, historia) in enumerate(historias_pool.items()):
    axes[0].plot(historia.history['val_accuracy'], label=nombre, linewidth=2, color=colores[i])
    axes[1].plot(historia.history['val_loss'], label=nombre, linewidth=2, color=colores[i])

axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs Metodo de Reduccion')
axes[0].legend()
axes[0].grid(True)

axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss vs Metodo de Reduccion')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Tabla
print("\n" + "=" * 65)
print("COMPARACION DE METODOS DE REDUCCION")
print("=" * 65)
print(f"{'Metodo':<20s} {'Accuracy':>10s} {'Parametros':>12s} {'Tiempo':>10s}")
print("-" * 52)
for nombre, res in resultados_pool.items():
    print(f"{nombre:<20s} {res['Accuracy']:>10.4f} {res['Parametros']:>12,} {res['Tiempo']:>10.1f}s")

### \u270d\ufe0f Pregunta - Experimento 5

**¿Cual metodo de reduccion de dimensionalidad funciono mejor? ¿Por que Max Pooling suele ser mas popular que Average Pooling? ¿Que ventajas y desventajas tiene usar stride en vez de pooling?**

*Pista: Max Pooling selecciona las activaciones mas fuertes (las caracteristicas mas relevantes detectadas), mientras que Average Pooling las diluye al promediar.*

### \u270d\ufe0f Tu respuesta:

*Escribe aqui tu respuesta...*

---
### Experimento 6: Visualizar Feature Maps

Una de las formas mas poderosas de entender que aprende una CNN es **visualizar los feature maps** (las salidas de cada capa convolucional). Vamos a construir una CNN de 3 bloques, entrenarla, y luego inspeccionar que ve en cada nivel.

In [ ]:
# ============================================================
# EXPERIMENTO 6: Visualizacion de feature maps
# ============================================================

# Construir una CNN de 3 bloques para este experimento
modelo_vis = keras.Sequential([
    # Bloque 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3),
                 name='conv2d_bloque1'),
    layers.MaxPooling2D((2, 2), name='maxpool_bloque1'),

    # Bloque 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2d_bloque2'),
    layers.MaxPooling2D((2, 2), name='maxpool_bloque2'),

    # Bloque 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv2d_bloque3'),
    layers.MaxPooling2D((2, 2), name='maxpool_bloque3'),

    # Clasificador
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='CNN_para_visualizacion')

modelo_vis.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Entrenar
print("Entrenando CNN de 3 bloques para visualizacion de feature maps...")
historia_vis = modelo_vis.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=25,
    batch_size=64,
    verbose=0,
    callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)

_, acc_vis = modelo_vis.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy en test: {acc_vis:.4f} ({acc_vis*100:.1f}%)")
print(f"Epocas entrenadas: {len(historia_vis.history['loss'])}")

In [ ]:
# Crear un modelo intermedio que devuelve las salidas de cada capa convolucional
# Esto nos permite "espiar" lo que ve la CNN en cada nivel

input_layer = keras.Input(shape=(32, 32, 3))
x = input_layer
salidas_intermedias = []
capas_conv = []

for capa in modelo_vis.layers:
    x = capa(x)
    if 'conv2d' in capa.name:
        salidas_intermedias.append(x)
        capas_conv.append(capa)

modelo_intermedio = keras.Model(inputs=input_layer, outputs=salidas_intermedias)

print("Capas convolucionales encontradas:")
for capa in capas_conv:
    print(f"  {capa.name}: {capa.output.shape}")

# Seleccionar una imagen de test para visualizar
idx_imagen = 3
img_test = X_test[idx_imagen:idx_imagen+1]  # Agregar dimension de batch
clase_real = nombres_clases[y_test[idx_imagen][0]]

# Obtener las activaciones (feature maps)
activaciones = modelo_intermedio.predict(img_test, verbose=0)

print(f"\nImagen seleccionada: clase '{clase_real}'")
print(f"Activaciones obtenidas:")
for i, act in enumerate(activaciones):
    print(f"  Capa {i+1} ({capas_conv[i].name}): shape = {act.shape}")

In [ ]:
# Visualizar la imagen original y los feature maps de cada capa
fig, axes = plt.subplots(1, 1, figsize=(3, 3))
axes.imshow(img_test[0])
axes.set_title(f'Imagen original: {clase_real}', fontsize=13)
axes.axis('off')
plt.tight_layout()
plt.show()

# Visualizar feature maps de la PRIMERA capa convolucional (32 filtros)
print("\n--- Feature Maps de la Capa 1 (32 filtros - detectan patrones SIMPLES) ---")
fig, axes = plt.subplots(4, 8, figsize=(18, 9))
for i in range(32):
    fila = i // 8
    col = i % 8
    axes[fila, col].imshow(activaciones[0][0, :, :, i], cmap='viridis')
    axes[fila, col].set_title(f'Filtro {i}', fontsize=8)
    axes[fila, col].axis('off')
plt.suptitle('Feature Maps - Capa 1 (32 filtros): bordes y colores basicos', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Feature maps de la SEGUNDA capa convolucional (64 filtros)
print("--- Feature Maps de la Capa 2 (64 filtros - detectan patrones INTERMEDIOS) ---")
fig, axes = plt.subplots(4, 8, figsize=(18, 9))
for i in range(32):  # Mostramos solo los primeros 32 de 64
    fila = i // 8
    col = i % 8
    axes[fila, col].imshow(activaciones[1][0, :, :, i], cmap='viridis')
    axes[fila, col].set_title(f'Filtro {i}', fontsize=8)
    axes[fila, col].axis('off')
plt.suptitle('Feature Maps - Capa 2 (primeros 32 de 64 filtros): texturas y formas', fontsize=14)
plt.tight_layout()
plt.show()

# Feature maps de la TERCERA capa convolucional (128 filtros)
print("--- Feature Maps de la Capa 3 (128 filtros - detectan patrones COMPLEJOS) ---")
fig, axes = plt.subplots(4, 8, figsize=(18, 9))
for i in range(32):  # Mostramos solo los primeros 32 de 128
    fila = i // 8
    col = i % 8
    axes[fila, col].imshow(activaciones[2][0, :, :, i], cmap='viridis')
    axes[fila, col].set_title(f'Filtro {i}', fontsize=8)
    axes[fila, col].axis('off')
plt.suptitle('Feature Maps - Capa 3 (primeros 32 de 128 filtros): partes de objetos', fontsize=14)
plt.tight_layout()
plt.show()

print("\nObservaciones:")
print("- Capa 1: los feature maps muestran bordes, colores y texturas basicas")
print("  (se pueden distinguir formas de la imagen original)")
print("- Capa 2: los patrones son mas abstractos, combinan bordes en formas")
print("  (la resolucion es menor: 16x16 -> 8x8)")
print("- Capa 3: los patrones son muy abstractos, dificiles de interpretar visualmente")
print("  (resolucion 4x4, representan partes complejas del objeto)")

### \u270d\ufe0f Pregunta - Experimento 6

**¿Que diferencias observas entre los feature maps de las primeras capas vs las ultimas? ¿Que tipo de patrones detecta cada nivel? ¿Por que los feature maps de las capas profundas son mas pequenos y abstractos?**

*Pista: Cada capa de MaxPooling reduce el tamano a la mitad. Las capas profundas combinan los patrones simples de las capas anteriores para formar representaciones mas complejas.*

### \u270d\ufe0f Tu respuesta:

*Escribe aqui tu respuesta...*

---
### Experimento 7: CNN optimizada con tecnicas de la Guia 03

Ahora vamos a construir nuestra mejor CNN aplicando las tecnicas de regularizacion y optimizacion que aprendimos en la Guia 03:
- **BatchNormalization** entre Conv2D y MaxPooling
- **Dropout** despues de las capas Dense
- **Data Augmentation** con ImageDataGenerator (rotacion, flip horizontal)
- **Early Stopping** y **ReduceLROnPlateau** como callbacks

In [ ]:
# ============================================================
# EXPERIMENTO 7: CNN optimizada con regularizacion y data augmentation
# ============================================================

# Modelo BASE (sin regularizacion, para comparar)
modelo_base = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='CNN_base')

modelo_base.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print(f"CNN base: {modelo_base.count_params():,} parametros")

# Modelo OPTIMIZADO (con todas las tecnicas)
modelo_opt = keras.Sequential([
    # Bloque 1 con BatchNormalization
    layers.Conv2D(32, (3, 3), padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),

    # Bloque 2 con BatchNormalization
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),

    # Bloque 3 con BatchNormalization
    layers.Conv2D(128, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),

    # Clasificador con Dropout
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
], name='CNN_optimizada')

modelo_opt.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print(f"CNN optimizada: {modelo_opt.count_params():,} parametros")

In [ ]:
# Configurar Data Augmentation
# ImageDataGenerator aplica transformaciones aleatorias a las imagenes
# durante el entrenamiento para generar mas variedad

datagen = ImageDataGenerator(
    rotation_range=15,          # Rotar hasta 15 grados
    width_shift_range=0.1,      # Desplazar horizontalmente hasta 10%
    height_shift_range=0.1,     # Desplazar verticalmente hasta 10%
    horizontal_flip=True,       # Voltear horizontalmente (espejo)
    zoom_range=0.1              # Zoom aleatorio hasta 10%
)

# Ajustar el generador a los datos de entrenamiento
datagen.fit(X_train)

# Visualizar ejemplos de data augmentation
fig, axes = plt.subplots(2, 6, figsize=(16, 6))

# Imagen original (fila superior)
img_aug = X_train[0]
for i in range(6):
    axes[0, i].imshow(img_aug)
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Original', fontsize=11)

# Versiones aumentadas (fila inferior)
for i in range(6):
    img_augmentada = datagen.random_transform(img_aug)
    axes[1, i].imshow(img_augmentada)
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Aumentada', fontsize=11)

plt.suptitle('Data Augmentation: misma imagen con transformaciones aleatorias', fontsize=14)
plt.tight_layout()
plt.show()

print("Data Augmentation configurada:")
print("  - Rotacion: hasta 15 grados")
print("  - Desplazamiento horizontal/vertical: hasta 10%")
print("  - Flip horizontal: Si (las imagenes siguen siendo reconocibles al voltearlas)")
print("  - Zoom: hasta 10%")
print("\nNota: NO usamos flip vertical porque un auto o avion al reves no es natural.")

In [ ]:
# Entrenar modelo BASE (sin regularizacion ni data augmentation)
print("Entrenando CNN BASE (sin regularizacion)...")
print("=" * 50)

historia_base = modelo_base.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    verbose=0,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    ]
)

_, acc_base = modelo_base.evaluate(X_test, y_test, verbose=0)
print(f"CNN base - Accuracy en test: {acc_base:.4f} ({acc_base*100:.1f}%)")
print(f"Epocas entrenadas: {len(historia_base.history['loss'])}")

In [ ]:
# Entrenar modelo OPTIMIZADO (con BN + Dropout + Data Augmentation + callbacks)
print("\nEntrenando CNN OPTIMIZADA (con regularizacion + data augmentation)...")
print("=" * 50)

# Callbacks para el modelo optimizado
callbacks_opt = [
    callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

# Entrenar CON data augmentation usando datagen.flow()
historia_opt = modelo_opt.fit(
    datagen.flow(X_train, y_train, batch_size=64),
    steps_per_epoch=len(X_train) // 64,
    validation_data=(X_val, y_val),
    epochs=80,
    verbose=0,
    callbacks=callbacks_opt
)

_, acc_opt = modelo_opt.evaluate(X_test, y_test, verbose=0)
print(f"CNN optimizada - Accuracy en test: {acc_opt:.4f} ({acc_opt*100:.1f}%)")
print(f"Epocas entrenadas: {len(historia_opt.history['loss'])}")

In [ ]:
# Comparar resultados
print("\n" + "=" * 60)
print("COMPARACION: CNN BASE vs CNN OPTIMIZADA")
print("=" * 60)
print(f"{'Metrica':<30s} {'CNN Base':>12s} {'CNN Optimizada':>15s}")
print("-" * 57)
print(f"{'Accuracy test':<30s} {acc_base:>12.4f} {acc_opt:>15.4f}")
print(f"{'Mejora absoluta':<30s} {'':>12s} {acc_opt - acc_base:>+15.4f}")
print(f"{'Mejora relativa':<30s} {'':>12s} {(acc_opt - acc_base)/acc_base*100:>+14.1f}%")
print(f"{'Epocas entrenadas':<30s} {len(historia_base.history['loss']):>12d} {len(historia_opt.history['loss']):>15d}")

# Graficar comparacion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(historia_base.history['val_accuracy'], label='CNN Base', linewidth=2, color='#e74c3c')
axes[0].plot(historia_opt.history['val_accuracy'], label='CNN Optimizada', linewidth=2, color='#2ecc71')
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy en Validacion')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(historia_base.history['val_loss'], label='CNN Base', linewidth=2, color='#e74c3c')
axes[1].plot(historia_opt.history['val_loss'], label='CNN Optimizada', linewidth=2, color='#2ecc71')
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss en Validacion')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Grafico de sobreajuste (train vs val) para cada modelo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CNN Base - sobreajuste
axes[0].plot(historia_base.history['accuracy'], label='Train', linewidth=2, linestyle='--')
axes[0].plot(historia_base.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('CNN Base: Train vs Validation\n(sobreajuste visible)')
axes[0].legend()
axes[0].grid(True)

# CNN Optimizada - menos sobreajuste
axes[1].plot(historia_opt.history['accuracy'], label='Train', linewidth=2, linestyle='--')
axes[1].plot(historia_opt.history['val_accuracy'], label='Validation', linewidth=2)
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('CNN Optimizada: Train vs Validation\n(sobreajuste reducido)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### \u270d\ufe0f Pregunta - Experimento 7

**¿Cuanto mejoro el modelo con las tecnicas de regularizacion y data augmentation? ¿Cual tecnica crees que tuvo mayor impacto? ¿Observas diferencias en el sobreajuste entre el modelo base y el optimizado?**

*Pista: Compara las curvas de train vs validation de ambos modelos. Un modelo que sobreajusta tendra una brecha grande entre train y validation accuracy.*

### \u270d\ufe0f Tu respuesta:

*Escribe aqui tu respuesta...*

---
## 5. Sintesis y Reflexion Final

Has completado una guia extensa sobre Redes Neuronales Convolucionales. Es hora de reflexionar sobre lo aprendido y consolidar los conceptos clave.

### \u270d\ufe0f Pregunta de Sintesis 1

**Explica en tus propias palabras por que las CNN son superiores a las DNN para tareas de vision por computadora. Menciona al menos tres razones tecnicas concretas.**

### \u270d\ufe0f Tu respuesta:

*Escribe aqui tu respuesta...*

### \u270d\ufe0f Pregunta de Sintesis 2

**Dibuja (describe textualmente) la arquitectura ideal que usarias para clasificar radiografias medicas en 5 categorias (Normal, Neumonia, COVID, Tuberculosis, Otro). Justifica cada capa que incluyas: por que ese numero de filtros, por que ese tamano, por que esa regularizacion.**

*Considera: las radiografias son imagenes en escala de grises de 224x224 pixeles. Es un problema medico donde los falsos negativos son peligrosos.*

### \u270d\ufe0f Tu respuesta:

*Escribe aqui tu respuesta...*

### \u270d\ufe0f Pregunta de Sintesis 3

**¿Que limitaciones crees que tienen las CNN que hemos construido en esta guia? ¿Que harias para mejorarlas? Menciona al menos 3 limitaciones y 3 posibles soluciones.**

*Pista: Piensa en el tamano del dataset, la resolucion de las imagenes, la profundidad de las redes, y conceptos como Transfer Learning (que veremos en la proxima guia).*

### \u270d\ufe0f Tu respuesta:

*Escribe aqui tu respuesta...*

---
## 6. Reto Extra

### Construir la mejor CNN posible para CIFAR-10

Aplica todo lo aprendido en esta guia para construir la **mejor CNN que puedas** para CIFAR-10.

**Objetivo:** Superar el **80% de accuracy** en el conjunto de test.

**Herramientas a tu disposicion:**
- Multiples bloques convolucionales (experimenta con la profundidad)
- Diferentes tamanios de filtro
- BatchNormalization
- Dropout (en capas densas y/o convolucionales)
- Data Augmentation (ImageDataGenerator)
- Callbacks: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
- Diferentes optimizadores y learning rates

**Instrucciones:**
1. Disena tu arquitectura CNN
2. Configura el data augmentation
3. Entrena con los callbacks adecuados
4. Evalua en el conjunto de test
5. **Documenta tus decisiones:** explica por que elegiste cada componente

**Tabla de referencia:**

| Accuracy en test | Nivel |
|:---:|:---:|
| < 70% | Basico |
| 70% - 75% | Intermedio |
| 75% - 80% | Avanzado |
| > 80% | Excelente |
| > 85% | Experto (muy dificil sin Transfer Learning) |

In [ ]:
# ============================================================
# RETO EXTRA: Tu mejor CNN para CIFAR-10
# ============================================================

# Tu codigo aqui: Define tu arquitectura CNN
# modelo_reto = keras.Sequential([
#     # ... tus capas aqui ...
# ])



In [ ]:
# Tu codigo aqui: Configura el data augmentation
# datagen_reto = ImageDataGenerator(
#     # ... tus transformaciones aqui ...
# )



In [ ]:
# Tu codigo aqui: Entrena tu modelo con callbacks
# historia_reto = modelo_reto.fit(
#     # ... tu entrenamiento aqui ...
# )



In [ ]:
# Tu codigo aqui: Evalua tu modelo y muestra los resultados
# _, acc_reto = modelo_reto.evaluate(X_test, y_test, verbose=0)
# print(f"Accuracy en test: {acc_reto:.4f} ({acc_reto*100:.1f}%)")



### \u270d\ufe0f Documenta tus decisiones de diseno:

**Arquitectura elegida:** *(describe las capas y justifica cada decision)*

**Data Augmentation:** *(que transformaciones usaste y por que)*

**Callbacks y optimizador:** *(que callbacks usaste y por que)*

**Resultado obtenido:** *(accuracy final y reflexion)*

*Escribe aqui tu respuesta...*

---
## 7. Referencias

1. **LeCun, Y., Bengio, Y., & Hinton, G.** (2015). Deep Learning. *Nature*, 521(7553), 436-444.
2. **Krizhevsky, A., Sutskever, I., & Hinton, G. E.** (2012). ImageNet Classification with Deep Convolutional Neural Networks. *NeurIPS 2012*.
3. **Chollet, F.** (2021). *Deep Learning with Python* (2nd Edition). Manning Publications.
4. **Geron, A.** (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd Edition). O'Reilly Media.
5. **Keras Documentation** - Conv2D Layer. https://keras.io/api/layers/convolution_layers/convolution2d/
6. **Keras Documentation** - Image preprocessing. https://keras.io/api/data_loading/image/
7. **CIFAR-10 Dataset** - Krizhevsky, A. https://www.cs.toronto.edu/~kriz/cifar.html
8. **Simonyan, K., & Zisserman, A.** (2015). Very Deep Convolutional Networks for Large-Scale Image Recognition (VGGNet). *ICLR 2015*.

---

**Fin de la Guia 05** | Siguiente: Guia 06 - Transfer Learning y Modelos Preentrenados

In [ ]:
print("Guia 05 completada. Buen trabajo!")
print("En la proxima guia aprenderemos sobre Transfer Learning,")
print("que nos permitira usar CNN preentrenadas para obtener resultados")
print("mucho mejores con menos datos y menos tiempo de entrenamiento.")